In [5]:
import csv

FILE_PATH = "birthday.csv"
MAX_ELEMENT = 200

class HeapNode:
    def __init__(self, key=0, name="", birthday=""):
        self.key = key
        self.name = name
        self.birthday = birthday

    def getKey(self):
        return self.key

class MaxHeap:
    def __init__(self):
        self.size = 0
        self.node = [HeapNode() for _ in range(MAX_ELEMENT)]

    def isEmpty(self): return self.size == 0
    def isFull(self): return self.size == MAX_ELEMENT - 1
    def getParent(self, i): return self.node[i // 2]
    def getLeft(self, i): return self.node[i * 2]
    def getRight(self, i): return self.node[i * 2 + 1]

    def insert(self, key, name, birthday):
        if self.isFull(): return
        self.size += 1
        idx = self.size
        while idx != 1 and key > self.getParent(idx).getKey():
            self.node[idx] = self.node[idx // 2]
            idx //= 2
        self.node[idx] = HeapNode(key, name, birthday)

    def remove(self):
        if self.isEmpty(): return None
        removed = self.node[1]
        last = self.node[self.size]
        self.size -= 1
        parent, child = 1, 2
        while child <= self.size:
            if child < self.size and self.node[child].getKey() < self.node[child + 1].getKey():
                child += 1
            if last.getKey() >= self.node[child].getKey(): break
            self.node[parent] = self.node[child]
            parent = child
            child *= 2
        self.node[parent] = last
        return removed

def load_birthdays(filepath):
    records = []
    try:
        with open(filepath, encoding="utf-8-sig", newline="") as f:
            reader = csv.reader(f)
            next(reader)
            for row in reader:
                # 1. 비어있거나 부족한 경우 건너뜀
                if not row or len(row) < 4:
                    continue
                
                # 2. 각 값이 비어있는지 확인 (' ' 공백 포함)
                if not row[0].strip() or not row[1].strip() or not row[2].strip() or not row[3].strip():
                    print(f"  [경고] 데이터 누락 발견: {row[0]} -> 건너뜀")
                    continue
                
                try:
                    name = row[0].strip()
                    # 숫자로 변환
                    year = int(float(row[1]))
                    month = int(float(row[2]))
                    day = int(float(row[3]))
                    
                    key = year * 10000 + month * 100 + day
                    bday = f"{year}년 {month:02d}월 {day:02d}일"
                    records.append((key, name, bday))
                except ValueError:
                    # 숫자가 아닌 문자(ex: "모름")가 들어있는 경우 처리
                    print(f"  [경고] 잘못된 숫자 형식: {row}")[cite: 1]
                    continue
    except FileNotFoundError:
        print(f"오류: '{filepath}' 파일을 찾을 수 없습니다.")
    return records

# 실행부
records = load_birthdays(FILE_PATH)
if records:
    heap = MaxHeap()
    for r in records:
        heap.insert(r[0], r[1], r[2])

    print(f"{'순위':<4} {'이름':<10} {'생년월일'}")
    print("-" * 30)
    for i in range(1, min(11, heap.size + 1)):
        node = heap.remove()
        print(f"{i:>2}위   {node.name:<10} {node.birthday}")

  [경고] 데이터 누락 발견: 유지민 -> 건너뜀
순위   이름         생년월일
------------------------------
 1위   이윤서        2006년 12월 27일
 2위   정희원        2006년 12월 21일
 3위   김효린        2006년 12월 16일
 4위   이예은        2006년 12월 09일
 5위   김주영        2006년 11월 20일
 6위   전은빈        2006년 11월 07일
 7위   이하연        2006년 09월 22일
 8위   김우현        2006년 09월 01일
 9위   최윤지        2006년 08월 30일
10위   유가현        2006년 08월 22일


In [7]:
import csv

GROUP_ROW_START = 33
GROUP_ROW_END   = 44


class BirthdayMissingError(Exception):
    pass


class Node:
    def __init__(self, name, birthday):
        self.name     = name
        self.birthday = birthday   # 문자열 or None
        self.prev     = None
        self.next     = None


class CircularDoublyLinkedList:
    def __init__(self):
        self.head = None
        self.size = 0

    def is_empty(self):
        return self.head is None

    def insert_tail(self, name, birthday):
        new_node = Node(name, birthday)
        if self.is_empty():
            new_node.prev = new_node
            new_node.next = new_node
            self.head = new_node
        else:
            tail          = self.head.prev
            tail.next     = new_node
            new_node.prev = tail
            new_node.next = self.head
            self.head.prev = new_node
        self.size += 1

    def display_forward(self):
        # head -> tail
        if self.is_empty():
            print("  (리스트가 비어 있습니다)")
            return
        cur  = self.head
        rank = 1
        while True:
            bday_str = cur.birthday if cur.birthday else "생년월일 없음"
            print(f"  {rank:>2}. {cur.name:<12} {bday_str}")
            rank += 1
            cur = cur.next
            if cur is self.head:
                break

    def display_reverse(self):
        # tail -> head
        if self.is_empty():
            print("  (리스트가 비어 있습니다)")
            return
        cur  = self.head.prev   # tail
        rank = 1
        while True:
            bday_str = cur.birthday if cur.birthday else "생년월일 없음"
            print(f"  {rank:>2}. {cur.name:<12} {bday_str}")
            rank += 1
            cur = cur.prev
            if cur is self.head.prev:
                break


def load_group_members(filepath, row_start, row_end):
    members = []
    with open(filepath, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        for cell_row, row in enumerate(reader, start=1):
            if cell_row < row_start or cell_row > row_end:
                continue
            if not row or row[0].strip() == "":
                continue
            name = row[0].strip()
            try:
                if len(row) < 4 or row[1].strip() == "" or row[2].strip() == "" or row[3].strip() == "":
                    raise BirthdayMissingError(f"'{name}': 생년월일 데이터 없음")
                year  = int(float(row[1]))
                month = int(float(row[2]))
                day   = int(float(row[3]))
                bday  = f"{year}년 {month:02d}월 {day:02d}일"
                members.append((name, bday))
            except BirthdayMissingError as e:
                print(f"  [예외처리] {e} -> birthday=None 으로 저장")
                members.append((name, None))
            except ValueError:
                print(f"  [예외처리] '{name}': 형식 오류 -> birthday=None 으로 저장")
                members.append((name, None))
    return members


# 메인
def main():
    filepath = "birthday.csv"

    members = load_group_members(filepath, GROUP_ROW_START, GROUP_ROW_END)

    if not members:
        print("조원 데이터를 불러오지 못했습니다.")
        return

    cdll = CircularDoublyLinkedList()
    for name, bday in members:
        cdll.insert_tail(name, bday)

    print(f"\n  총 {cdll.size}명 리스트 삽입 완료")

    print(f"\n  ▶ 순방향 출력")
    print("  " + "-" * 46)
    cdll.display_forward()

    print(f"\n  ◀ 역방향 출력")
    print("  " + "-" * 46)
    cdll.display_reverse()

    print("=" * 50)


if __name__ == "__main__":
    main()



  총 12명 리스트 삽입 완료

  ▶ 순방향 출력
  ----------------------------------------------
   1. 황다원          2004년 10월 15일
   2. 박다인          2005년 01월 19일
   3. 안예원          2004년 07월 09일
   4. 정서하          2006년 07월 20일
   5. 김우현          2006년 09월 01일
   6. 양시언          2006년 04월 18일
   7. 유가현          2006년 08월 22일
   8. 이하연          2006년 09월 22일
   9. 장예은          2006년 03월 17일
  10. 전은빈          2006년 11월 07일
  11. 정세희          2004년 01월 27일
  12. 이예은          2006년 12월 09일

  ◀ 역방향 출력
  ----------------------------------------------
   1. 이예은          2006년 12월 09일
   2. 정세희          2004년 01월 27일
   3. 전은빈          2006년 11월 07일
   4. 장예은          2006년 03월 17일
   5. 이하연          2006년 09월 22일
   6. 유가현          2006년 08월 22일
   7. 양시언          2006년 04월 18일
   8. 김우현          2006년 09월 01일
   9. 정서하          2006년 07월 20일
  10. 안예원          2004년 07월 09일
  11. 박다인          2005년 01월 19일
  12. 황다원          2004년 10월 15일
